# 04 - Looking at actual recommendations

Metric tables hide what the lists look like. This notebook runs the final pipeline for a few test users and puts its output next to popularity, MF and the graph models, then looks at the cases the error analysis flagged.

Everything here uses the saved checkpoints (`checkpoints/`) and the pipeline configuration chosen on validation users.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from recsysx.config import load_config
from recsysx.data import RecData
from recsysx.experiments.artifacts import load_topk
from recsysx.pipeline import RecommendationPipeline

cfg = load_config()
data = RecData.load(cfg)
pipe = RecommendationPipeline.from_artifacts(cfg, data)
titles, genres = data.items["title"].to_numpy(), data.items["genres"].to_numpy()
test_users = data.eval_users("test")
print("retriever:", pipe.retriever_key, "| ranker:", pipe.ranker_variant, "| candidates:", pipe.n_candidates, "| MMR lambda:", pipe.mmr_lambda)

retriever: hybrid | ranker: temporal | candidates: 200 | MMR lambda: 0.9


In [2]:
def show_user(u, k=10):
    row = data.users.iloc[u]
    tr = data.train_ratings
    liked = tr[(tr["user_idx"] == u) & (tr["rating"] >= 4)].sort_values("timestamp")["item_idx"].to_numpy()
    targets = set(data.targets[u].indices.tolist())
    print(f"user {u}: {row['gender']}, age group {row['age']}, {data.group[u]}, "
          f"{len(liked)} liked movies before the cutoff, {len(targets)} liked after")
    for i in liked[-5:][::-1]:
        print("   liked:", titles[i], "|", genres[i])
    rec = pipe.recommend(int(u), k=k)
    j = int(np.where(test_users == u)[0][0])
    lists = {"final pipeline": rec.items}
    for key, label in (("popularity", "popularity"), ("mf", "MF-BPR"), ("lightgcn", "LightGCN"), ("graphsage", "GraphSAGE")):
        lists[label] = load_topk(cfg, key, 42, "test")[1][j, :k].tolist()
    table = pd.DataFrame({name: [titles[i] + (" *" if i in targets else "") for i in items] for name, items in lists.items()})
    table.index = range(1, k + 1)
    print("(* = the user liked it after the cutoff)")
    print(f"pipeline latency: {rec.timings_ms['total_ms']:.1f} ms")
    return table

## A warm user

In [3]:
warm = [u for u in test_users if data.group[u] == "warm" and np.diff(data.targets.indptr)[u] >= 15]
show_user(warm[0])

user 634: M, age group 56, warm, 29 liked movies before the cutoff, 129 liked after
   liked: Papillon (1973) | Drama
   liked: Straight Story, The (1999) | Drama
   liked: Nashville (1975) | Drama|Musical
   liked: Fear and Loathing in Las Vegas (1998) | Comedy|Drama
   liked: Midnight Cowboy (1969) | Drama
(* = the user liked it after the cutoff)
pipeline latency: 32.4 ms


,final pipeline,popularity,MF-BPR,LightGCN,GraphSAGE
1,Almost Famous (2000),American Beauty (1999) *,American Beauty (1999) *,American Beauty (1999) *,American Beauty (1999) *
2,Saving Private Ryan (1998),Star Wars: Episode IV - A New Hope (1977),Schindler's List (1993),Being John Malkovich (1999) *,Being John Malkovich (1999) *
3,One Flew Over the Cuckoo's Nest (1975),Star Wars: Episode V - The Empire Strikes Back...,Fargo (1996) *,Shakespeare in Love (1998) *,Shakespeare in Love (1998) *
4,Shakespeare in Love (1998) *,Saving Private Ryan (1998),Shakespeare in Love (1998) *,Fargo (1996) *,"Shawshank Redemption, The (1994)"
5,L.A. Confidential (1997),Raiders of the Lost Ark (1981),North by Northwest (1959) *,"Shawshank Redemption, The (1994)",Fargo (1996) *
6,Casablanca (1942),"Silence of the Lambs, The (1991) *",Casablanca (1942),"Sixth Sense, The (1999)",Schindler's List (1993)
7,Schindler's List (1993),"Matrix, The (1999)",Being John Malkovich (1999) *,Schindler's List (1993),"Full Monty, The (1997)"
8,"Silence of the Lambs, The (1991) *",Star Wars: Episode VI - Return of the Jedi (1983),"Shawshank Redemption, The (1994)","Silence of the Lambs, The (1991) *",Rushmore (1998)
9,Being John Malkovich (1999) *,"Sixth Sense, The (1999)","Silence of the Lambs, The (1991) *",Election (1999),Babe (1995) *
10,Good Will Hunting (1997) *,Fargo (1996) *,Election (1999),L.A. Confidential (1997),Election (1999)


## A sparse user

In [4]:
sparse = [u for u in test_users if data.group[u] == "sparse" and np.diff(data.targets.indptr)[u] >= 10]
show_user(sparse[0])

user 663: M, age group 35, sparse, 15 liked movies before the cutoff, 106 liked after
   liked: L.A. Confidential (1997) | Crime|Film-Noir|Mystery|Thriller
   liked: Silence of the Lambs, The (1991) | Drama|Thriller
   liked: Gandhi (1982) | Drama
   liked: Raising Arizona (1987) | Comedy
   liked: Star Wars: Episode V - The Empire Strikes Back (1980) | Action|Adventure|Drama|Sci-Fi|War
(* = the user liked it after the cutoff)
pipeline latency: 18.7 ms


,final pipeline,popularity,MF-BPR,LightGCN,GraphSAGE
1,Braveheart (1995) *,American Beauty (1999) *,American Beauty (1999) *,American Beauty (1999) *,American Beauty (1999) *
2,Terminator 2: Judgment Day (1991) *,Star Wars: Episode IV - A New Hope (1977) *,Braveheart (1995) *,"Sixth Sense, The (1999) *",Shakespeare in Love (1998) *
3,Star Wars: Episode IV - A New Hope (1977) *,"Matrix, The (1999)","Sixth Sense, The (1999) *",Star Wars: Episode IV - A New Hope (1977) *,"Matrix, The (1999)"
4,Erin Brockovich (2000) *,Star Wars: Episode VI - Return of the Jedi (1983),Star Wars: Episode IV - A New Hope (1977) *,Terminator 2: Judgment Day (1991) *,Star Wars: Episode VI - Return of the Jedi (1983)
5,Star Wars: Episode VI - Return of the Jedi (1983),"Sixth Sense, The (1999) *",Fargo (1996) *,Braveheart (1995) *,Star Wars: Episode IV - A New Hope (1977) *
6,"Godfather, The (1972)",Fargo (1996) *,"Matrix, The (1999)","Godfather, The (1972)",Braveheart (1995) *
7,"Matrix, The (1999)",Terminator 2: Judgment Day (1991) *,Being John Malkovich (1999) *,"Shawshank Redemption, The (1994)","Sixth Sense, The (1999) *"
8,Frequency (2000),"Shawshank Redemption, The (1994)",Terminator 2: Judgment Day (1991) *,Star Wars: Episode VI - Return of the Jedi (1983),"Princess Bride, The (1987) *"
9,Jurassic Park (1993) *,Braveheart (1995) *,"Shawshank Redemption, The (1994)",Shakespeare in Love (1998) *,Toy Story 2 (1999) *
10,"Princess Bride, The (1987) *","Godfather, The (1972)",Pulp Fiction (1994) *,"Matrix, The (1999)","Godfather, The (1972)"


## A cold user (no history at all)

In [5]:
cold = [u for u in test_users if data.group[u] == "cold"]
show_user(cold[0])

user 3: M, age group 45, cold, 0 liked movies before the cutoff, 18 liked after
(* = the user liked it after the cutoff)
pipeline latency: 18.6 ms


,final pipeline,popularity,MF-BPR,LightGCN,GraphSAGE
1,Star Wars: Episode IV - A New Hope (1977) *,American Beauty (1999),American Beauty (1999),Star Wars: Episode IV - A New Hope (1977) *,American Beauty (1999)
2,L.A. Confidential (1997),Star Wars: Episode IV - A New Hope (1977) *,Star Wars: Episode V - The Empire Strikes Back...,American Beauty (1999),Star Wars: Episode IV - A New Hope (1977) *
3,Saving Private Ryan (1998) *,Star Wars: Episode V - The Empire Strikes Back...,Star Wars: Episode IV - A New Hope (1977) *,Terminator 2: Judgment Day (1991),Star Wars: Episode V - The Empire Strikes Back...
4,American Beauty (1999),Saving Private Ryan (1998) *,Raiders of the Lost Ark (1981) *,Raiders of the Lost Ark (1981) *,"Silence of the Lambs, The (1991)"
5,Fargo (1996),Raiders of the Lost Ark (1981) *,Saving Private Ryan (1998) *,Star Wars: Episode V - The Empire Strikes Back...,Fargo (1996)
6,Star Wars: Episode V - The Empire Strikes Back...,"Silence of the Lambs, The (1991)",Fargo (1996),"Silence of the Lambs, The (1991)",Saving Private Ryan (1998) *
7,"Godfather, The (1972)","Matrix, The (1999)","Matrix, The (1999)","Godfather, The (1972)",Raiders of the Lost Ark (1981) *
8,Raiders of the Lost Ark (1981) *,Star Wars: Episode VI - Return of the Jedi (1983),"Silence of the Lambs, The (1991)",Star Wars: Episode VI - Return of the Jedi (1983),Schindler's List (1993)
9,Back to the Future (1985),"Sixth Sense, The (1999)","Sixth Sense, The (1999)",Saving Private Ryan (1998) *,"Sixth Sense, The (1999)"
10,Terminator 2: Judgment Day (1991),Fargo (1996),Star Wars: Episode VI - Return of the Jedi (1983),"Sixth Sense, The (1999)",Star Wars: Episode VI - Return of the Jedi (1983)


In [6]:
# how different are the lists of two random cold users? (only demographics differ)
a, b = cold[1], cold[2]
ra, rb = pipe.recommend(int(a)).items, pipe.recommend(int(b)).items
print(data.users.iloc[a][["gender", "age", "occupation"]].to_dict(), data.users.iloc[b][["gender", "age", "occupation"]].to_dict())
print("shared items in the two top-10 lists:", len(set(ra) & set(rb)))

{'gender': 'F', 'age': 50, 'occupation': 9}

 {'gender': 'M', 'age': 25, 'occupation': 17}
shared items in the two top-10 lists: 5


## Personalization in numbers

In [7]:
pd.read_csv("../results/tables/personalization_results.csv")[
    ["model", "personalization@10", "mean_jaccard_with_popularity_list", "distinct_lists", "coverage@10", "novelty@10"]].round(3)

,model,personalization@10,mean_jaccard_with_popularity_list,distinct_lists,coverage@10,novelty@10
0,Popularity,0.608,1.000,473,0.032,1.624
1,Popularity (recent),0.598,0.599,470,0.026,1.691
2,Popularity (demographic),0.752,0.589,531,0.046,1.706
3,ItemKNN,0.791,0.536,561,0.092,1.947
4,MF-BPR,0.779,0.539,561,0.141,2.031
5,Two-Tower,0.946,0.184,785,0.210,2.655
6,Node2Vec,0.815,0.433,561,0.206,2.694
7,GraphSAGE,0.817,0.403,561,0.190,2.299
8,GAT,0.822,0.341,561,0.202,2.333
9,LightGCN,0.797,0.381,560,0.158,2.130


What the examples above show in numbers. For users with a history the pipeline's list really depends on the user (the warm drama fan gets One Flew Over the Cuckoo's Nest, Casablanca and Schindler's List instead of Star Wars). For cold users the lists are close to the popularity list: two cold users with completely different demographics still share half of their top 10. In the table, the popularity lists differ between users only through the removal of already-rated movies (personalization 0.61). MF, ItemKNN, Node2Vec, GraphSAGE and GAT produce exactly 561 distinct lists for 881 test users because all 321 cold users get the same list. The two-tower model and the hybrid use demographics, so they separate cold users as well.

## Rule-picked failure cases (from `scripts/run_analysis.py`)

In [8]:
display(Markdown(open("../results/tables/error_examples.md", encoding="utf-8").read()))

# Error examples (test users)

Generated by `scripts/run_analysis.py`. Each case is selected by a rule stated in its heading.

## no model gets a single hit in the top 20 (most test positives among such users): user_idx 45 (cold)

- no positive ratings before the cutoff
- test positives: 33, mean popularity 95, 27% tail items, 3% cold items
- most common genres in the test positives: Horror, Thriller, Mystery
- recall@20: Popularity 0.00, MF-BPR 0.00, GraphSAGE 0.00, Hybrid (id+content+graph) 0.00, Final pipeline 0.00
- final pipeline: 9% of the test positives were in the 200 retrieved candidates

Final pipeline top 10:

1. American Beauty (1999) — Comedy|Drama
2. Star Wars: Episode IV - A New Hope (1977) — Action|Adventure|Fantasy|Sci-Fi
3. Saving Private Ryan (1998) — Action|Drama|War
4. Matrix, The (1999) — Action|Sci-Fi|Thriller
5. Braveheart (1995) — Action|Drama|War
6. Star Wars: Episode V - The Empire Strikes Back (1980) — Action|Adventure|Drama|Sci-Fi|War
7. Terminator 2: Judgment Day (1991) — Action|Sci-Fi|Thriller
8. Pulp Fiction (1994) — Crime|Drama
9. Star Wars: Episode VI - Return of the Jedi (1983) — Action|Adventure|Romance|Sci-Fi|War
10. Raiders of the Lost Ark (1981) — Action|Adventure

A few of the test positives:

1. Wes Craven's New Nightmare (1994) — Horror
2. Halloween: The Curse of Michael Myers (1995) — Horror|Thriller
3. Evil Dead II (Dead By Dawn) (1987) — Action|Adventure|Comedy|Horror
4. Amityville Horror, The (1979) — Horror
5. Nightmare on Elm Street, A (1984) — Horror

## largest GraphSAGE loss against MF-BPR: user_idx 2597 (sparse)

- liked before cutoff: 3 movies, their mean popularity is 869 training positives
- test positives: 12, mean popularity 1470, 0% tail items, 0% cold items
- most common genres in the test positives: Action, Sci-Fi, Thriller
- most common genres in the training likes: Action, Adventure, Thriller
- recall@20: Popularity 0.42, MF-BPR 0.42, GraphSAGE 0.17, Hybrid (id+content+graph) 0.58, Final pipeline 0.33
- final pipeline: 100% of the test positives were in the 200 retrieved candidates

Final pipeline top 10:

1. Terminator 2: Judgment Day (1991) — Action|Sci-Fi|Thriller
2. Braveheart (1995) — Action|Drama|War
3. Fugitive, The (1993) — Action|Thriller
4. Saving Private Ryan (1998) — Action|Drama|War
5. Gladiator (2000) — Action|Drama
6. Sixth Sense, The (1999) — Thriller
7. Matrix, The (1999) — Action|Sci-Fi|Thriller
8. Raiders of the Lost Ark (1981) — Action|Adventure **(hit)**
9. Hunt for Red October, The (1990) — Action|Thriller
10. Star Wars: Episode V - The Empire Strikes Back (1980) — Action|Adventure|Drama|Sci-Fi|War **(hit)**

A few of the test positives:

1. Star Wars: Episode IV - A New Hope (1977) — Action|Adventure|Fantasy|Sci-Fi
2. Silence of the Lambs, The (1991) — Drama|Thriller
3. Godfather, The (1972) — Action|Crime|Drama
4. Die Hard (1988) — Action|Thriller
5. Star Wars: Episode V - The Empire Strikes Back (1980) — Action|Adventure|Drama|Sci-Fi|War

## narrowest taste among warm users (largest share of likes in one genre): user_idx 1896 (warm)

- liked before cutoff: 108 movies, their mean popularity is 371 training positives
- test positives: 294, mean popularity 405, 3% tail items, 1% cold items
- most common genres in the test positives: Drama, Comedy, Romance
- most common genres in the training likes: Drama, Romance, Comedy
- recall@20: Popularity 0.03, MF-BPR 0.03, GraphSAGE 0.02, Hybrid (id+content+graph) 0.04, Final pipeline 0.04
- final pipeline: 26% of the test positives were in the 200 retrieved candidates

Final pipeline top 10:

1. Chicken Run (2000) — Animation|Children's|Comedy **(hit)**
2. Three Colors: Red (1994) — Drama
3. Shakespeare in Love (1998) — Comedy|Romance **(hit)**
4. L.A. Confidential (1997) — Crime|Film-Noir|Mystery|Thriller **(hit)**
5. Being John Malkovich (1999) — Comedy **(hit)**
6. Run Lola Run (Lola rennt) (1998) — Action|Crime|Romance
7. Sixth Sense, The (1999) — Thriller **(hit)**
8. Life Is Beautiful (La Vita è bella) (1997) — Comedy|Drama
9. Saving Private Ryan (1998) — Action|Drama|War
10. Sweet Hereafter, The (1997) — Drama **(hit)**

A few of the test positives:

1. Toy Story (1995) — Animation|Children's|Comedy
2. Persuasion (1995) — Romance
3. Usual Suspects, The (1995) — Crime|Thriller
4. Indian in the Cupboard, The (1995) — Adventure|Children's|Fantasy
5. Apollo 13 (1995) — Drama

## most active user (most training positives): user_idx 4276 (warm)

- liked before cutoff: 1209 movies, their mean popularity is 276 training positives
- test positives: 226, mean popularity 127, 19% tail items, 2% cold items
- most common genres in the test positives: Drama, Comedy, Romance
- most common genres in the training likes: Drama, Thriller, Action
- recall@20: Popularity 0.04, MF-BPR 0.04, GraphSAGE 0.04, Hybrid (id+content+graph) 0.04, Final pipeline 0.03
- final pipeline: 27% of the test positives were in the 200 retrieved candidates

Final pipeline top 10:

1. Diner (1982) — Comedy|Drama
2. Fly, The (1958) — Horror|Sci-Fi
3. Toy Story (1995) — Animation|Children's|Comedy **(hit)**
4. Cape Fear (1962) — Film-Noir|Thriller
5. Jerry Maguire (1996) — Drama|Romance **(hit)**
6. Rounders (1998) — Crime|Drama **(hit)**
7. North by Northwest (1959) — Drama|Thriller
8. Cider House Rules, The (1999) — Drama **(hit)**
9. Them! (1954) — Sci-Fi|Thriller|War
10. Awakenings (1990) — Drama

A few of the test positives:

1. Toy Story (1995) — Animation|Children's|Comedy
2. Nixon (1995) — Drama
3. Two Bits (1995) — Drama
4. Crossing Guard, The (1995) — Drama
5. Anne Frank Remembered (1995) — Documentary

## long-tail test positives that no model ranks in any user's top 20 (most frequent)

1. Bamboozled (2000) — Comedy
2. Tigerland (2000) — Drama
3. Broken Hearts Club, The (2000) — Drama
4. Joe Gould's Secret (2000) — Drama
5. Washington Square (1997) — Drama

Across all test users 84% of the test positives are not in the top 20 of any model.


The rule-picked cases show the typical failure modes: a cold user with a niche taste (horror) that no input can reveal, a heavy user whose remaining likes are mostly less popular movies, and a sparse user whose future likes are exactly the blockbusters, where popularity and MF are right and GraphSAGE moved away from them. The last block lists long-tail movies nobody recommends: mostly 2000 releases that were tail items only because they were new.

## Retrieval misses vs. ranking misses

In [9]:
pd.read_csv("../results/tables/error_pipeline_breakdown.csv").round(3)

,item_bucket,hit (top-20),not retrieved,ranked below 20,targets
0,cold (0 train positives),0.000,0.969,0.031,65
1,head (top 20%),0.117,0.452,0.431,40982
2,mid,0.001,0.941,0.058,15473
3,tail (bottom 40%),0.000,0.998,0.002,1747


In [10]:
pd.read_csv("../results/tables/error_pipeline_breakdown_by_group.csv").round(3)

,user_group,hit (top-20),not retrieved,ranked below 20,targets
0,cold,0.077,0.608,0.315,29201
1,sparse,0.081,0.597,0.322,4296
2,warm,0.089,0.588,0.323,24770


Most misses happen before the ranker ever sees the movie. For head movies the misses are split roughly evenly between "not retrieved" and "retrieved but below rank 20", for everything else more than 94% are never retrieved. Improving the ranker cannot fix that, it would need a retriever that covers more of the catalog.